# Paper-like microlensing photometry and astrometry plots

Plot the G-band light curve, photometric residuals, and along-scan astrometric
residuals relative to the five-parameter and microlensing solutions. Export a
four-panel figure, a full-range diagnostic, selected measurements, complete
CCD/transit/photometry masks and checksummed provenance.

Run notebook 02 with CSV output for the catalogue model parameters. Run notebook
06 for the same `SOURCE_ID`, including its final epoch-file export cell.
This notebook reads those local Gaia DR4 FITS files and makes no archive requests.
Model and plotting helpers are embedded; no private checkout is required.


The data are described in the Gaia DR4 publication by **Wyrzykowski et al. (2026)**.

## What is actually selected?


1. **G photometry:** finite `g_obs_time`, `g_mag` and (when flux/error columns
   exist) finite positive magnitude error,
   $\sigma_G=(2.5/\ln 10)\,\sigma_F/F$. No photometric flag cut or statistical
   outlier clipping is applied by the production G plotting function.
   Its magnitude-axis limits come from the level-2 model, so some measurements
   can be outside the visible panel. The exports distinguish plotted rows from
   rows inside that display window. The full-range diagnostic reveals them.
2. **Astrometric CCDs:** keep finite time, scan angle, AL centroid/error and both
   AL/AC parallax factors. `used_by_agis_al` is read and reported, but is **not a
   selection criterion**. Other flags are retained without cuts. In particular,
   CCDs not used by AGIS can enter the average. This documents the existing
   figure workflow, not a recommendation for a new scientific fit.
3. **One point per transit:** inverse-variance weighted time and AL centroid;
   $\sigma_{\rm transit}=\sqrt{1/\sum w}$, with $w=1/\sigma_{\rm CCD}^2$.
   The production implementation assigns weight 1 to finite nonpositive CCD
   errors; we preserve and count that behavior explicitly. It averages model
   predictions **arithmetically** over retained CCDs, whereas data are weighted.
4. **Astrometric clipping:** residual $r=m_{\rm lens}-AL$ using the catalogue
   level-3 solution, not the 5p model. Among finite residuals with finite time
   and positive finite transit error, compute $c=\mathrm{median}(r)$ and
   $s=1.4826\,\mathrm{median}|r-c|$. If this scale is zero/nonfinite, use the RMS
   about the median. Reject $|r-c|>5s$. This is **one pass**, on residuals in mas,
   not error-normalised residuals. Use the same retained transits in both
   astrometric panels. The production fallback, if no points survive, keeps
   transits with finite time and positive finite error; this is recorded below.

There is no refit after clipping. These are **figure masks**, not evidence of
the internal CU7 fitting cuts. 

In [ ]:
from pathlib import Path
from typing import Any
from datetime import datetime, timezone
import hashlib
import json
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits
from astropy.time import Time
import astromet
from IPython.display import display

# Dependencies: numpy pandas matplotlib astropy astromet ipython.
# All model/plotting helpers are embedded below; no local ECLIPSE import needed.

def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "01_archive_access.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the gaia-dr4-microlensing-notebooks folder.")

ROOT = find_project_root(Path.cwd())

SOURCE_ID = '4048864055145726976'  # Exact decimal string; match notebook 06
assert SOURCE_ID.isascii() and SOURCE_ID.isdecimal() and 0 < int(SOURCE_ID) <= np.iinfo(np.int64).max
DATA_DIR = ROOT / 'data' / SOURCE_ID
PARAMETER_FILE = ROOT / 'outputs/gaiadr4/gaia_dr4_microlensing_events_joined.csv'
ASTROMETRY_SUPPLEMENT = DATA_DIR / 'astrometry_supplement.ecsv'  # Optional replacement catalogue astrometry
PHOT_FILE = DATA_DIR / 'epoch_photometry.fits'
ASTRO_FILE = DATA_DIR / 'epoch_astrometry.fits'
INPUT_PROVENANCE = DATA_DIR / 'provenance.json'  # Written by notebook 06
CANDIDATE_FILE = ROOT / 'data/event_names.csv'  # Optional publication labels
OUTPUT_DIR = ROOT / 'outputs/gaiadr4/paper_like_plots' / SOURCE_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_URL = 'https://gea.esac.esa.int/'
TAP_SCHEMA = 'gaiadr4'
DATA_RELEASE = 'Gaia DR4'
TIME_OFFSET_JD = 2455197.5
try:
    import cmasher  # optional: same colour map as production when installed
    CMAP = plt.get_cmap('cmr.neon')
except ImportError:
    CMAP = plt.get_cmap('viridis')

## Read epoch products and catalogue model parameters

Validate the source ID, release/schema and input checksums against the download
ledgers. FITS photometry IDs and flags are retained without floating-point ID
conversion. Astrometry remains an Astropy table with its vector columns and
masks. Some astrometry files store the source ID in the FITS `SOURCEID` header.

In [ ]:
def read_epoch_fits(path):
    with fits.open(path, memmap=False) as hdus:
        tables = [h for h in hdus if isinstance(h, (fits.BinTableHDU, fits.TableHDU))]
        if len(tables) != 1:
            raise ValueError(f'Expected one epoch table in {path}; inspect its FITS HDUs.')
        table = Table.read(tables[0])
        for name in list(table.colnames):
            if name.lower() != name:
                table.rename_column(name, name.lower())
        header_id = tables[0].header.get('SOURCEID', hdus[0].header.get('SOURCEID'))
        if header_id is not None:
            table.meta['source_id'] = str(header_id)
        return table

def verify_download_inputs():
    epoch_info = json.loads(INPUT_PROVENANCE.read_text())
    catalogue_ledger = PARAMETER_FILE.with_suffix(PARAMETER_FILE.suffix + '.provenance.json')
    catalogue_info = json.loads(catalogue_ledger.read_text())
    for info in (epoch_info, catalogue_info):
        if info.get('schema') != TAP_SCHEMA or info.get('data_release') != DATA_RELEASE:
            raise ValueError('Input release/schema differs from configuration; do not mix releases.')
    if epoch_info.get('source_id') != SOURCE_ID:
        raise ValueError('Epoch download ledger belongs to another source.')
    if hashlib.sha256(PARAMETER_FILE.read_bytes()).hexdigest() != catalogue_info['sha256']:
        raise ValueError('Catalogue checksum disagrees with its provenance ledger.')
    for path in (PHOT_FILE, ASTRO_FILE):
        expected = epoch_info.get('files', {}).get(path.name, {}).get('sha256')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f'Epoch checksum disagrees with download ledger: {path}')
    return epoch_info, catalogue_info


for path in (PARAMETER_FILE, PHOT_FILE, ASTRO_FILE, INPUT_PROVENANCE):
    if not path.is_file():
        raise FileNotFoundError(f"Set the input paths in the configuration cell: {path}")

epoch_provenance, catalogue_provenance = verify_download_inputs()
parameters = Table.read(PARAMETER_FILE, format='ascii.csv')
id_col = 'SOURCE_ID' if 'SOURCE_ID' in parameters.colnames else 'source_id'
matches = np.flatnonzero(np.asarray(parameters[id_col], dtype=np.int64) == int(SOURCE_ID))
if len(matches) != 1:
    raise ValueError(f'Expected exactly one parameter row for {SOURCE_ID}; found {len(matches)}')
selected_parameters = parameters[matches].copy()
supplement_applied = False
if ASTROMETRY_SUPPLEMENT.is_file():
    supplement = Table.read(ASTROMETRY_SUPPLEMENT)
    supplement_id = 'SOURCE_ID' if 'SOURCE_ID' in supplement.colnames else 'source_id'
    smatches = np.flatnonzero(np.asarray(supplement[supplement_id], dtype=np.int64) == int(SOURCE_ID))
    if len(smatches) > 1:
        raise ValueError('Duplicate source in astrometry supplement')
    if len(smatches) == 1:
        for name in ('ref_epoch', 'ra', 'ra_error', 'dec', 'dec_error', 'parallax',
                     'parallax_error', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error',
                     'ruwe', 'astrometric_gof_al', 'astrometric_excess_noise', 'visibility_periods_used'):
            selected_parameters[name][0] = supplement[name][smatches[0]]
        supplement_applied = True
row = selected_parameters[0]
phot_table = read_epoch_fits(PHOT_FILE)
photometry_columns = {}
for name in phot_table.colnames:
    column = phot_table[name]
    if name.endswith('_id') or name.endswith('_flags'):
        photometry_columns[name] = [str(value) for value in column]
    elif name.endswith(('_obs_time', '_mag', '_flux', '_flux_error')):
        photometry_columns[name] = np.asarray(np.ma.asarray(column, dtype=float).filled(np.nan))
    else:
        photometry_columns[name] = [None if np.ndim(value) == 0 and np.ma.is_masked(value) else value
                                    for value in column]
phot = pd.DataFrame(photometry_columns)
astro = read_epoch_fits(ASTRO_FILE)
if not {'g_obs_time', 'g_mag'}.issubset(phot.columns):
    raise ValueError('Expected per-transit photometry columns g_obs_time and g_mag')
required_astro = {'obs_time_tcb', 'scan_pos_angle', 'centroid_pos_al',
                  'centroid_pos_error_al', 'parallax_factor_al', 'parallax_factor_ac', 'used_by_agis_al'}
if missing := required_astro - set(astro.colnames):
    raise ValueError(f'Astrometry product layout differs: missing {sorted(missing)}')
for table in (phot_table, astro):
    if 'source_id' in table.colnames:
        identities = {str(x) for x in table['source_id']}
    else:
        identities = {str(table.meta.get('source_id', ''))}
    if identities != {SOURCE_ID}:
        raise ValueError('Epoch product source ID does not match selected source.')

label = SOURCE_ID
if CANDIDATE_FILE.is_file():
    candidates = pd.read_csv(CANDIDATE_FILE, dtype={'source_id': str, 'sourceid': str})
    name_id = 'source_id' if 'source_id' in candidates else 'sourceid'
    candidate = candidates.loc[candidates[name_id] == SOURCE_ID]
    if len(candidate) == 1:
        label = f"{candidate.iloc[0]['event']} / {SOURCE_ID}"
print(label)
print(f'{len(phot)} photometry rows; {len(astro)} astrometry transits; supplement applied: {supplement_applied}')
display(phot.head(3))

## Model equations and embedded helpers

The helpers implement time conversion, the Paczyński photometric model, CCD
averaging and the catalogue astrometric microlensing model. Photometry uses TCB
decimal year and the astrometric grid uses TCB Julian year. Astrometric models
are evaluated at CCD times before averaging. Original flags remain in `astro`.

`level3_astrometric_models` evaluates the five-parameter reference and the
Level-3 astrometric microlensing solution at the CCD epochs, then averages the
model predictions per transit. It uses the Paczyński-2 trajectory parameters
and Paczyński-3 source offsets, parallax, proper motions and Einstein radius.


In [ ]:
def rotationMatrix(phi):
    '''
    Parameters
    ----------
    phi : float
        Angle in radians.

    Returns
    -------
    R : np.array (2,2)
        Rotation matrix of angle phi.
    '''
    R = np.array([[np.cos(phi),-np.sin(phi)],[np.sin(phi),np.cos(phi)]])
    return R


def rotateVectors( vec , angle ):
    '''
    Parameters
    ----------
    vec : np.array (2,n)
        Array of vectors to rotate.
    angle : float
        Angle in radians.

    Returns
    -------
    np.array (2,n)
        Arrays of input vectors rotated by given angle.
    '''
    R = rotationMatrix(angle)
    return R @ vec


def compute_delta_taubeta( piE, phi, sc_a, fw, fz ):
    '''
    Compute the effect of parallax on tau/beta.

    Parameters
    ----------
    piE : float
        Microlensing parallax - length of the vector piE.
    phi : float
        Angle of the vector piE measured from North.
    sc_a : float np.array
        Scan angles at observation times. Defines the angle of direction fw measured from North.
    fw : float np.array
        Parallax factors AL (or any direction, but consistent with sc_a) at observation times.
    fz : float np.array
        Parallax factors AC (or any direction, but consistent with sc_a) at observation times.

    Returns
    -------
    delta_tb : 2xN float np.array
        The modification of tau beta from the parallax effect on both source and lens.

    '''
    # Construct the microlensing parallax effect
    n = len( fw )
    delta_tb = np.array([piE*rotationMatrix( sc_a[i] - phi )@np.array([fw[i],-fz[i]]) for i in range(n) ]).T

    return delta_tb


def compute_taubeta( t0, tE, u0 , time, piE, phi, sc_a, fw, fz ):
    '''
    Compute tau/beta using only Gaia data (scan angle and parallax factor AL and AC).
    This can also compute tau beta at any point using sc_a with only zeros and fw/fz the computed parallax factors in corresponding directions.

    Parameters
    ----------
    t0 : float
        t0 in the barycentric reference frame (Julian Year, time at barycenter).
    tE : float
        Event timescale in the barycentric reference frame (years, time at barycenter).
    u0 : float
        Impact parameter in units of thetaE in the barycentric reference frame.
    time : float np.array.
        Times of the observations (Julian Year, time at barycenter).
    piE : float
        Microlensing parallax - length of the vector piE.
    phi : float
        Angle of the vector piE measured from North.
    sc_a : float np.array
        Scan angles at observation times. Defines the angle of direction fw measured from North.
    fw : float np.array
        Parallax factors AL (or any direction, but consistent with sc_a) at observation times.
    fz : float np.array
        Parallax factors AC (or any direction, but consistent with sc_a) at observation times.

    Returns
    -------
    tb : 2xN float np.array
        tau/beat values in tau/beta coordinate system.

    '''
    # First contruct the part not including parallax
    tb = np.array([(time-t0)/tE,u0*np.ones_like(time)])

    # If parallax included, add contribution
    delta_tb = compute_delta_taubeta(piE, phi, sc_a, fw, fz)
    tb += delta_tb

    return tb


def compute_u_NE( t0, tE, u0 , time, piE, phi, sc_a, fw, fz ):
    '''
    Compute u in the NE coordinate system. First compute tau/beta and then rotate.

    Parameters
    ----------
    t0 : float
        t0 in the barycentric reference frame (Julian Year, time at barycenter).
    tE : float
        Event timescale in the barycentric reference frame (years, time at barycenter).
    u0 : float
        Impact parameter in units of thetaE in the barycentric reference frame.
    time : float np.array.
        Times of the observations (Julian Year, time at barycenter).
    piE : float
        Microlensing parallax - length of the vector piE.
    phi : float
        Angle of the vector piE measured from North.
    sc_a : float np.array
        Scan angles at observation times. Defines the angle of direction fw measured from North.
    fw : float np.array
        Parallax factors AL (or any direction, but consistent with sc_a) at observation times.
    fz : float np.array
        Parallax factors AC (or any direction, but consistent with sc_a) at observation times.

    Returns
    -------
    u_NE : 2xN float np.array
        u in the North-East corrdinate system.

    '''
    # Compute the taubeta value
    taubeta = compute_taubeta( t0, tE, u0, time, piE, phi, sc_a, fw, fz )
    u_NE = rotateVectors( taubeta, phi - np.pi )

    return u_NE


def shift_from_u( u:np.array ) -> np.array:
    '''
    Compute the astrometric microlensing shift (in units of theta_E) from vector u.

    Parameters
    ----------
    u : np.array
        position of the source relative to the lens.

    Returns
    -------
    2xN float np.array
        Shift of the centroid for the image of the source in units of theta_E in the same coordinate system as u input.

    '''
    return u / (np.linalg.norm(u, axis=0)**2 + 2)


def make_5p_matrix( time_wrt_refepoch:np.array , sc_a:np.array, fw:np.array ) -> np.array:
    """ Make the matrix which transform parameters from the 5p to the observable along scan.
    Args  :
        time_wrt_refepoch : 1D Array with the timestamps with respect to reference epoch (at which w=0 in data)
        sc_a : 1D Array with the scan angle [rad] at the corresponding timestamps
        fw : 1D Parallax factor along scan at the corresponding timestamps
    Returns :
        np.array : Transform the 5p to AL using 5p model
    """

    # Make sin(sc_a) & cos(sc_a)
    s = np.sin(sc_a)
    c = np.cos(sc_a)

    # Make the array
    A = np.array([s,c,fw,s*time_wrt_refepoch,c*time_wrt_refepoch])

    return A.T


def make_5p_thetaE_matrix( time_wrt_refepoch:np.array , sc_a:np.array, fw:np.array , u_NE:np.array ) -> np.array:
    """ Make the matrix which transform parameters from the 5p+thetaE to the observable : along scan.
    Args  :
        time_wrt_refepoch : 1D Array with the timestamps with respect to reference epoch (at which w=0 in data)
        sc_a : 1D Array with the scan angle [rad] at the corresponding timestamps
        fw : 1D Parallax factor along scan at the corresponding timestamps
    Returns :
        np.array : Transform the 5p+thetaE to AL using 5p+thetaE model
    """

    # No microlensing part
    A_5p = make_5p_matrix( time_wrt_refepoch, sc_a, fw ).T

    # Shift from u
    delta_u_vec = shift_from_u( u_NE )
    delta_u = np.cos(sc_a)*delta_u_vec[0,:] + np.sin(sc_a)*delta_u_vec[1,:]

    # Make whole array
    A_5p_thetaE = np.array([A_5p[0], A_5p[1] ,A_5p[2], A_5p[3], A_5p[4], delta_u])

    return A_5p_thetaE.T


def linear_model_matrix(am_data, t0, tE, u0, piE, phi, model='5p+lens'):

    # Load the parameters
    time, timeRel, sc_a, fw, fz, refEpoch = am_data['ts'], am_data['ts'] - am_data['ref'], am_data['sc_a'], am_data['fw'], am_data['fz'], am_data['ref']
    # wError = np.sqrt( wError**2 + excess_noise**2 )

    if ( model == '5p'):
        # Construct linear model without lensing
        R = make_5p_matrix( timeRel, sc_a, fw )
    elif ( model == '5p+lens' ):
        # Construct lensed linear model
        u_NE = compute_u_NE( t0, tE, u0 , time, piE, phi, sc_a, fw, fz )
        R = make_5p_thetaE_matrix( timeRel, sc_a, fw , u_NE)
    else:
        logging.warning('Wrong model name')
        return None

    return R

In [ ]:
def scalar(value: Any, default: float = np.nan) -> float:
    try:
        if hasattr(value, "filled"):
            value = value.filled(np.nan)
        if np.ma.is_masked(value):
            return default
        return float(value)
    except Exception:
        return default


def row_value(row: Any, name: str, default: float = np.nan) -> float:
    if name not in row.colnames:
        return default
    return scalar(row[name], default=default)


def masked_float_array(value: Any, fill: float = np.nan) -> np.ndarray:
    return np.ma.asarray(value).astype(float).filled(fill)


def days_to_decimal_year(times: np.ndarray) -> np.ndarray:
    return Time(np.asarray(times, dtype=float) + TIME_OFFSET_JD, format="jd", scale="tcb").decimalyear


def paczynski_magnification(u: np.ndarray) -> np.ndarray:
    return (u * u + 2.0) / (u * np.sqrt(u * u + 4.0))


def parallax_tau_beta(times: np.ndarray, row: Any, level: int) -> tuple[np.ndarray, np.ndarray] | None:
    u0 = row_value(row, f"paczynski{level}_u0")
    te = row_value(row, f"paczynski{level}_te")
    tmax = row_value(row, f"paczynski{level}_tmax")
    if not np.all(np.isfinite([u0, te, tmax])) or te == 0:
        return None
    if level == 2:
        pi_east = row_value(row, "paczynski2_parallax_east")
        pi_north = row_value(row, "paczynski2_parallax_north")
        ra = row_value(row, "ra")
        dec = row_value(row, "dec")
        if np.all(np.isfinite([pi_east, pi_north, ra, dec])):
            time_year = Time(np.asarray(times, dtype=float) + TIME_OFFSET_JD, format="jd", scale="tcb").decimalyear
            t0_year = Time(tmax + TIME_OFFSET_JD, format="jd", scale="tcb").decimalyear
            te_year = te / 365.25
            pi_e = float(np.hypot(pi_east, pi_north))
            phi = float(np.arctan2(pi_east, pi_north))

            params = astromet.params()
            params.ra = ra
            params.dec = dec
            params.parallax = 1
            params.pmrac = 0
            params.pmdec = 0
            params.epoch = 2017.5

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                f_east, f_north = astromet.track(time_year, params)
            with warnings.catch_warnings(), np.errstate(all="ignore"):
                warnings.simplefilter("ignore", category=RuntimeWarning)
                u_ne = compute_u_NE(
                    t0_year,
                    te_year,
                    u0,
                    time_year,
                    pi_e,
                    phi,
                    np.zeros_like(time_year),
                    f_north,
                    -f_east,
                )
            return u_ne[0], u_ne[1]

    tau = (times - tmax) / te
    beta = np.full_like(tau, u0, dtype=float)
    return tau, beta


def paczynski_mag_model(times: np.ndarray, row: Any, band: str, level: int) -> np.ndarray | None:
    mag0 = row_value(row, f"paczynski{level}_{band}0")
    tau_beta = parallax_tau_beta(times, row, level)
    if not np.isfinite(mag0) or tau_beta is None:
        return None
    tau, beta = tau_beta
    u = np.sqrt(tau * tau + beta * beta)
    amplification = paczynski_magnification(u)
    if level == 1:
        fs = row_value(row, f"paczynski1_fs_{band}", 1.0)
        if not np.isfinite(fs):
            fs = 1.0
        flux_factor = fs * amplification + (1.0 - fs)
        if np.any(flux_factor <= 0):
            flux_factor = np.where(flux_factor > 0, flux_factor, np.nan)
        return mag0 - 2.5 * np.log10(flux_factor)
    return mag0 - 2.5 * np.log10(amplification)


def magnitude_errors(df: pd.DataFrame, prefix: str) -> np.ndarray | None:
    flux_col = f"{prefix}_flux"
    err_col = f"{prefix}_flux_error"
    if flux_col not in df or err_col not in df:
        return None
    flux = df[flux_col].to_numpy(float)
    flux_err = df[err_col].to_numpy(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        mag_err = 2.5 / np.log(10.0) * flux_err / flux
    mag_err[~np.isfinite(mag_err) | (mag_err <= 0)] = np.nan
    return mag_err


def event_year_markers(row: Any) -> tuple[float, float, float]:
    tmax = row_value(row, "paczynski2_tmax")
    te = row_value(row, "paczynski2_te")
    if not np.all(np.isfinite([tmax, te])):
        return np.nan, np.nan, np.nan
    t0 = float(Time(tmax + TIME_OFFSET_JD, format="jd", scale="tcb").decimalyear)
    t_e = float(te / 365.25)
    return t0, t_e, float(np.sqrt(2.0) * t_e)


def add_event_markers(ax: plt.Axes, row: Any, x_min: float, x_max: float) -> None:
    t0, _, delta_t = event_year_markers(row)
    if not np.isfinite(t0):
        return
    ax.axvline(x=t0, ls=":", c="m", lw=2, alpha=0.4, zorder=0)
    if np.isfinite(delta_t):
        for sign in (-1, 1):
            t_line = t0 + sign * delta_t
            if x_min <= t_line <= x_max:
                ax.axvline(x=t_line, ls=":", c="m", lw=1.5, alpha=0.25, zorder=0)


def event_marker_handles() -> list[mpl.lines.Line2D]:
    return [
        mpl.lines.Line2D([0], [0], color="m", ls=":", lw=2, alpha=0.4, label=r"$t_{\rm max}^{\rm level2}$"),
        mpl.lines.Line2D([0], [0], color="m", ls=":", lw=1.5, alpha=0.25, label=r"$t_{\rm max}^{\rm level2}\pm\sqrt{2}t_E^{\rm level2}$"),
    ]


def photometry_time_span(df: pd.DataFrame) -> tuple[float, float] | None:
    if "g_obs_time" not in df or "g_mag" not in df:
        return None
    t_days = df["g_obs_time"].to_numpy(float)
    mag = df["g_mag"].to_numpy(float)
    mask = np.isfinite(t_days) & np.isfinite(mag)
    if not np.any(mask):
        return None
    t_year = days_to_decimal_year(t_days[mask])
    return finite_span(t_year, (float(np.nanmin(t_year)), float(np.nanmax(t_year))))


def merged_time_xlim(*spans: tuple[float, float] | None) -> tuple[float, float] | None:
    finite_spans = [
        (lo, hi)
        for span in spans
        if span is not None
        for lo, hi in [span]
        if np.isfinite(lo) and np.isfinite(hi)
    ]
    if not finite_spans:
        return None
    lo = min(span[0] for span in finite_spans)
    hi = max(span[1] for span in finite_spans)
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if lo == hi:
        return lo - 0.5, hi + 0.5
    pad = 0.03 * (hi - lo)
    return lo - pad, hi + pad


def to_epoch_days(obs_time_tcb: Any) -> np.ndarray:
    arr = masked_float_array(obs_time_tcb)
    return arr / 86400e9


def to_epoch_years(obs_time_tcb: Any) -> np.ndarray:
    days = to_epoch_days(obs_time_tcb)
    return Time(days + TIME_OFFSET_JD, format="jd", scale="tcb").jyear


def resize_to_length(value: Any, length: int) -> np.ndarray:
    arr = np.atleast_1d(masked_float_array(value))
    if len(arr) == 0:
        return np.full(length, np.nan)
    return np.resize(arr, length)


def astrometry_points(table: Table) -> dict[str, np.ndarray]:
    time_grid: list[np.ndarray] = []
    scan_grid: list[np.ndarray] = []
    pf_al_grid: list[np.ndarray] = []
    pf_ac_grid: list[np.ndarray] = []
    centroid_grid: list[np.ndarray] = []
    centroid_err_grid: list[np.ndarray] = []
    times_binned: list[float] = []
    scan_binned: list[float] = []
    pf_al_binned: list[float] = []
    pf_ac_binned: list[float] = []
    al: list[float] = []
    al_err: list[float] = []
    used: list[bool] = []
    for row in table:
        row_times = to_epoch_years(row["obs_time_tcb"])
        row_scan = np.deg2rad(masked_float_array(row["scan_pos_angle"]))
        row_al = masked_float_array(row["centroid_pos_al"])
        row_err = masked_float_array(row["centroid_pos_error_al"])
        row_used = np.ma.asarray(row["used_by_agis_al"]).filled(False).astype(bool)
        n = min(len(row_times), len(row_scan), len(row_al), len(row_err), len(row_used))
        if n == 0:
            continue
        row_pf_al = resize_to_length(row["parallax_factor_al"], n)
        row_pf_ac = resize_to_length(row["parallax_factor_ac"], n)
        row_times = row_times[:n]
        row_scan = row_scan[:n]
        row_al = row_al[:n]
        row_err = row_err[:n]
        row_used = row_used[:n]
        row_pf_al = row_pf_al[:n]
        row_pf_ac = row_pf_ac[:n]
        good = (
            np.isfinite(row_times)
            & np.isfinite(row_scan)
            & np.isfinite(row_al)
            & np.isfinite(row_err)
            & np.isfinite(row_pf_al)
            & np.isfinite(row_pf_ac)
        )
        if not np.any(good):
            continue
        weights = np.where(row_err[good] > 0, 1.0 / row_err[good] ** 2, 1.0)
        time_grid.append(row_times[good])
        scan_grid.append(row_scan[good])
        pf_al_grid.append(row_pf_al[good])
        pf_ac_grid.append(row_pf_ac[good])
        centroid_grid.append(row_al[good])
        centroid_err_grid.append(row_err[good])
        times_binned.append(float(np.average(row_times[good], weights=weights)))
        scan_binned.append(float(np.average(row_scan[good], weights=weights)))
        pf_al_binned.append(float(np.average(row_pf_al[good], weights=weights)))
        pf_ac_binned.append(float(np.average(row_pf_ac[good], weights=weights)))
        al.append(float(np.average(row_al[good], weights=weights)))
        al_err.append(float(np.sqrt(1.0 / np.sum(weights))))
        used.append(bool(np.any(row_used[good])))
    return {
        "time": np.asarray(times_binned),
        "scan_angle": np.asarray(scan_binned),
        "parallax_factor_al": np.asarray(pf_al_binned),
        "parallax_factor_ac": np.asarray(pf_ac_binned),
        "al": np.asarray(al),
        "al_err": np.asarray(al_err),
        "used": np.asarray(used, dtype=bool),
        "time_grid": np.asarray(time_grid, dtype=object),
        "scan_angle_grid": np.asarray(scan_grid, dtype=object),
        "parallax_factor_al_grid": np.asarray(pf_al_grid, dtype=object),
        "parallax_factor_ac_grid": np.asarray(pf_ac_grid, dtype=object),
        "centroid_grid": np.asarray(centroid_grid, dtype=object),
        "centroid_err_grid": np.asarray(centroid_err_grid, dtype=object),
    }


def astrometry_time_span(table: Table) -> tuple[float, float] | None:
    values: list[np.ndarray] = []
    for row in table:
        row_times = to_epoch_years(row["obs_time_tcb"])
        row_times = row_times[np.isfinite(row_times)]
        if len(row_times):
            values.append(row_times)
    if not values:
        return None
    times = np.concatenate(values)
    return finite_span(times, (float(np.nanmin(times)), float(np.nanmax(times))))


def flatten_object_arrays(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    arrays = [np.asarray(value, dtype=float) for value in values]
    lengths = np.asarray([len(value) for value in arrays], dtype=int)
    if not arrays:
        return np.asarray([], dtype=float), lengths
    return np.concatenate(arrays), lengths


def mean_per_transit(values: np.ndarray, lengths: np.ndarray) -> np.ndarray:
    out: list[float] = []
    start = 0
    for length in lengths:
        chunk = values[start : start + length]
        out.append(float(np.nanmean(chunk)) if len(chunk) else np.nan)
        start += length
    return np.asarray(out)


def astrometric_model_geometry(row: Any) -> tuple[float, float, float, float, float] | None:
    tmax = row_value(row, "paczynski2_tmax")
    te = row_value(row, "paczynski2_te")
    u0 = row_value(row, "paczynski2_u0")
    pi_east = row_value(row, "paczynski2_parallax_east")
    pi_north = row_value(row, "paczynski2_parallax_north")
    if not np.all(np.isfinite([tmax, te, u0, pi_east, pi_north])) or te == 0:
        return None
    t0 = Time(tmax + TIME_OFFSET_JD, format="jd", scale="tcb").decimalyear
    t_e = te / 365.25
    pi_e = float(np.hypot(pi_east, pi_north))
    phi = float(np.arctan2(pi_east, pi_north))
    return float(t0), float(t_e), float(u0), pi_e, phi


def level3_astrometric_models(points: dict[str, np.ndarray], row: Any) -> tuple[np.ndarray | None, np.ndarray | None]:
    geometry = astrometric_model_geometry(row)
    if geometry is None:
        return None, None
    t0, t_e, u0, pi_e, phi = geometry
    ts, lengths = flatten_object_arrays(points["time_grid"])
    sc_a, _ = flatten_object_arrays(points["scan_angle_grid"])
    fw, _ = flatten_object_arrays(points["parallax_factor_al_grid"])
    fz, _ = flatten_object_arrays(points["parallax_factor_ac_grid"])
    if len(ts) == 0 or not (len(ts) == len(sc_a) == len(fw) == len(fz)):
        return None, None
    ref_epoch = row_value(row, "ref_epoch", 2017.5)
    if not np.isfinite(ref_epoch):
        ref_epoch = 2017.5
    am_data = {"ts": ts, "sc_a": sc_a, "fw": fw, "fz": fz, "ref": ref_epoch}

    model_5p = None
    a5 = np.asarray([0.0, 0.0, row_value(row, "parallax"), row_value(row, "pmra"), row_value(row, "pmdec")])
    if np.all(np.isfinite(a5)):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            r5 = linear_model_matrix(am_data, t0, t_e, u0, pi_e, phi, "5p")
        if r5 is not None:
            with np.errstate(all="ignore"):
                model_5p = mean_per_transit(r5 @ a5, lengths)

    model_lens = None
    a6 = np.asarray(
        [
            row_value(row, "paczynski3_delta_ra_source"),
            row_value(row, "paczynski3_delta_dec_source"),
            row_value(row, "paczynski3_varpi_source"),
            row_value(row, "paczynski3_pmra_source"),
            row_value(row, "paczynski3_pmdec_source"),
            row_value(row, "paczynski3_einstein_radius"),
        ]
    )
    if np.all(np.isfinite(a6)):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            r6 = linear_model_matrix(am_data, t0, t_e, u0, pi_e, phi, "5p+lens")
        if r6 is not None:
            with np.errstate(all="ignore"):
                model_lens = mean_per_transit(r6 @ a6, lengths)
    return model_5p, model_lens


def finite_span(values: np.ndarray, fallback: tuple[float, float]) -> tuple[float, float]:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return fallback
    return float(np.nanmin(finite)), float(np.nanmax(finite))

## Explicit selection masks and exports

`astrometry_selection` below contains the clipping block copied verbatim from
`plot_astrometry`, so the audit and the production panels use the same rule.
`ccd_audit` expands every original CCD slot, retaining all supplied fields,
flags and the input row/CCD index. Its `used_in_transit_average` column records
the finite-value mask. `transit_audit` records the second, transit-level mask.
No original download is overwritten.

In [ ]:
def astrometry_selection(points, row):
    model_5p, model_lens = level3_astrometric_models(points, row)
    if model_lens is None:
        raise ValueError("No finite level-3 astrometric lens model; cannot audit its clipping.")
    residuals_5p = model_5p - points["al"] if model_5p is not None else np.full_like(points["al"], np.nan)
    residuals_lens = model_lens - points["al"] if model_lens is not None else np.full_like(points["al"], np.nan)
    res_for_clip = np.asarray(residuals_lens, dtype=float)
    finite_mask = np.isfinite(points["time"]) & np.isfinite(points["al_err"]) & (points["al_err"] > 0) & np.isfinite(res_for_clip)
    if finite_mask.any():
        median_res = np.nanmedian(res_for_clip[finite_mask])
        mad = np.nanmedian(np.abs(res_for_clip[finite_mask] - median_res))
        mad_sigma = 1.4826 * mad if np.isfinite(mad) else np.nan
        rms_res = np.sqrt(np.nanmean((res_for_clip[finite_mask] - median_res) ** 2))
        sigma = mad_sigma if np.isfinite(mad_sigma) and mad_sigma > 0 else rms_res
        outlier_mask = np.abs(res_for_clip - median_res) > 5.0 * sigma if np.isfinite(sigma) and sigma > 0 else np.zeros_like(res_for_clip, dtype=bool)
    else:
        outlier_mask = np.zeros_like(res_for_clip, dtype=bool)
    keep = finite_mask & ~outlier_mask
    if not np.any(keep):
        keep = np.isfinite(points["time"]) & np.isfinite(points["al_err"]) & (points["al_err"] > 0)

    return residuals_5p, residuals_lens, finite_mask, outlier_mask, keep

In [ ]:
points = astrometry_points(astro)
res5, res_lens, finite_mask, outlier_mask, keep = astrometry_selection(points, row)
fallback = not np.any(finite_mask & ~outlier_mask)

# Reconstruct CCD membership while keeping every original slot and flag.
ccd_records, transit_records = [], []
point_index = 0
for raw_index, raw in enumerate(astro):
    times = to_epoch_years(raw['obs_time_tcb'])
    scan = np.deg2rad(masked_float_array(raw['scan_pos_angle']))
    al = masked_float_array(raw['centroid_pos_al'])
    err = masked_float_array(raw['centroid_pos_error_al'])
    used = np.ma.asarray(raw['used_by_agis_al']).filled(False).astype(bool)
    n = min(map(len, (times, scan, al, err, used)))
    pfa = resize_to_length(raw['parallax_factor_al'], n)
    pfc = resize_to_length(raw['parallax_factor_ac'], n)
    good = (np.isfinite(times[:n]) & np.isfinite(scan[:n]) & np.isfinite(al[:n])
            & np.isfinite(err[:n]) & np.isfinite(pfa) & np.isfinite(pfc))
    has_point = bool(np.any(good))
    for j in range(max(np.ma.asarray(raw[name]).size for name in required_astro)):
        record = {'input_row': raw_index, 'ccd_index': j}
        for name in astro.colnames:
            value = np.ma.asarray(raw[name])
            item = value if value.ndim == 0 else (value[j] if j < len(value) else np.ma.masked)
            record[name] = None if np.ma.is_masked(item) else (str(item) if name.endswith('_id') else item.item())
        in_average = j < n and bool(good[j])
        record.update(used_in_transit_average=in_average,
                      transit_kept_for_figure=bool(keep[point_index]) if has_point else False,
                      used_in_retained_transit=in_average and has_point and bool(keep[point_index]))
        ccd_records.append(record)
    if not has_point:
        transit_records.append({
            'input_row': raw_index, 'transit_id': str(raw['transit_id']),
            'point_index': None, 'n_ccd_in_average': 0,
            'n_ccd_used_by_agis_in_average': 0, 'n_ccd_nonpositive_error_in_average': 0,
            'any_ccd_used_by_agis': False, 'finite_for_clipping': False,
            'five_sigma_outlier': False, 'kept_for_figure': False,
            'fallback_applied': fallback, 'reason': 'no finite CCDs for averaging',
        })
        continue
    i = point_index
    transit_records.append({
        'input_row': raw_index, 'transit_id': str(raw['transit_id']), 'point_index': i,
        'time_jyear_tcb': points['time'][i], 'al_mas': points['al'][i],
        'al_error_mas': points['al_err'][i], 'residual_5p_model_minus_data_mas': res5[i],
        'residual_lens_model_minus_data_mas': res_lens[i],
        'n_ccd_in_average': int(good.sum()),
        'n_ccd_used_by_agis_in_average': int(np.sum(used[:n] & good)),
        'n_ccd_nonpositive_error_in_average': int(np.sum(good & (err[:n] <= 0))),
        'any_ccd_used_by_agis': bool(points['used'][i]),
        'finite_for_clipping': bool(finite_mask[i]), 'five_sigma_outlier': bool(outlier_mask[i]),
        'kept_for_figure': bool(keep[i]), 'fallback_applied': fallback,
        'reason': ('retained' if keep[i] else 'five-sigma outlier' if finite_mask[i] and outlier_mask[i]
                   else 'nonfinite residual/time or nonpositive error'),
    })
    point_index += 1
assert point_index == len(points['time'])
ccd_audit = pd.DataFrame(ccd_records)
transit_audit = pd.DataFrame(transit_records)

phot_audit = phot.copy()
phot_audit.insert(0, 'input_row', np.arange(len(phot)))
mag_err = magnitude_errors(phot, 'g')
phot_keep = np.isfinite(phot.g_obs_time.to_numpy(float)) & np.isfinite(phot.g_mag.to_numpy(float))
if mag_err is not None:
    phot_keep &= np.isfinite(mag_err)
    phot_audit['g_mag_error'] = mag_err
phot_audit['kept_for_g_plot'] = phot_keep
phot_model = paczynski_mag_model(phot.g_obs_time.to_numpy(float), row, 'g', 2)
if phot_model is None:
    raise ValueError('Selected source has no complete level-2 G model')
phot_audit['residual_model_minus_data_mag'] = phot_model - phot.g_mag.to_numpy(float)

valid_res = res_lens[finite_mask]
center = float(np.median(valid_res)) if len(valid_res) else np.nan
mad_sigma = float(1.4826 * np.median(np.abs(valid_res-center))) if len(valid_res) else np.nan
rms = float(np.sqrt(np.mean((valid_res-center)**2))) if len(valid_res) else np.nan
scale = mad_sigma if np.isfinite(mad_sigma) and mad_sigma > 0 else rms
summary = {
    'photometry_input': len(phot), 'photometry_kept': int(phot_keep.sum()),
    'astrometry_input_transits': len(astro), 'astrometry_input_ccd_slots': len(ccd_audit),
    'ccd_in_transit_averages': int(ccd_audit.used_in_transit_average.sum()),
    'ccd_not_used_by_agis_in_averages': int((transit_audit.n_ccd_in_average - transit_audit.n_ccd_used_by_agis_in_average).sum()),
    'ccd_nonpositive_error_in_averages': int(transit_audit.n_ccd_nonpositive_error_in_average.sum()),
    'astrometry_averaged_transits': len(points['time']),
    'astrometry_finite_for_clipping': int(finite_mask.sum()),
    'astrometry_outliers_among_finite': int((finite_mask & outlier_mask).sum()),
    'astrometry_kept': int(keep.sum()), 'fallback_applied': fallback,
    'residual_median_mas': center, 'robust_sigma_mas': scale,
    'scale_method': '1.4826 MAD' if np.isfinite(mad_sigma) and mad_sigma > 0 else 'RMS about median',
    'five_sigma_threshold_mas': 5 * scale,
}
display(pd.Series(summary, name='selection audit'))
display(transit_audit.loc[~transit_audit.kept_for_figure])

## Four-panel paper-like plots

Both astrometric panels use the same lens-based clipping mask. A separate
full-range diagnostic exposes rejected transits and photometric measurements
outside the main figure's display window. No refit is performed after clipping.

In [ ]:
def plot_photometry(
    ax: plt.Axes,
    residual_ax: plt.Axes,
    df: pd.DataFrame,
    row: Any,
    xlim: tuple[float, float] | None = None,
) -> None:
    if "g_obs_time" not in df or "g_mag" not in df:
        ax.text(0.5, 0.5, "No G photometry", transform=ax.transAxes, ha="center", va="center")
        ax.set_axis_off()
        residual_ax.set_axis_off()
        return
    t_days = df["g_obs_time"].to_numpy(float)
    mag = df["g_mag"].to_numpy(float)
    mag_err = magnitude_errors(df, "g")
    mask = np.isfinite(t_days) & np.isfinite(mag)
    if mag_err is not None:
        mask &= np.isfinite(mag_err)
    if not np.any(mask):
        ax.text(0.5, 0.5, "No G photometry", transform=ax.transAxes, ha="center", va="center")
        ax.set_axis_off()
        residual_ax.set_axis_off()
        return
    t_days_used = t_days[mask]
    t_year = days_to_decimal_year(t_days_used)
    mag_used = mag[mask]
    err_used = mag_err[mask] if mag_err is not None else None
    ax.scatter(t_year, mag_used, c=t_year, edgecolor="k", cmap=CMAP, alpha=1.0, s=40, zorder=4)
    if err_used is not None:
        ax.errorbar(t_year, mag_used, yerr=err_used, fmt="none", c="k", zorder=3, snap=False)
    tgrid_days = np.linspace(float(np.nanmin(t_days_used)), float(np.nanmax(t_days_used)), 1000)
    tgrid_year = days_to_decimal_year(tgrid_days)
    model = paczynski_mag_model(tgrid_days, row, "g", 2)
    h_mod = None
    if model is not None and np.any(np.isfinite(model)):
        (h_mod,) = ax.plot(tgrid_year, model, c="m", lw=2.5, zorder=1, label="Level2 microlensing model")
        finite_model = model[np.isfinite(model)]
        ax.set_ylim(np.nanmin(finite_model) - 0.12, np.nanmax(finite_model) + 0.07)
    x_min, x_max = xlim if xlim is not None else (float(np.nanmin(tgrid_year)), float(np.nanmax(tgrid_year)))
    ax.set_xlim(x_min, x_max)
    residual_ax.set_xlim(x_min, x_max)
    add_event_markers(ax, row, x_min, x_max)
    ax.invert_yaxis()
    ax.set_title("Photometric lensing")
    ax.set_xlabel("")
    ax.set_ylabel(r"$G$")
    legend_handles = []
    if h_mod is not None:
        legend_handles.append(h_mod)
    legend_handles.extend(event_marker_handles())
    ax.legend(fontsize=8, handles=legend_handles)

    model_at_data = paczynski_mag_model(t_days_used, row, "g", 2)
    if model_at_data is not None:
        res_vals = model_at_data - mag_used
        residual_ax.scatter(t_year, res_vals, c=t_year, edgecolor="k", cmap=CMAP, alpha=1.0, s=40, zorder=4)
        if err_used is not None:
            residual_ax.errorbar(t_year, res_vals, yerr=err_used, fmt="none", c="k", zorder=3, snap=False)
    residual_ax.axhline(y=0, ls="-", c="m", lw=2, alpha=0.6, zorder=1)
    add_event_markers(residual_ax, row, x_min, x_max)
    residual_ax.set_xlabel(r"Time [year]")
    residual_ax.set_ylabel(r"$\Delta G$")


def plot_astrometry(
    rax_5: plt.Axes,
    rax_lens: plt.Axes,
    table_or_path: Table | Path,
    row: Any,
    xlim: tuple[float, float] | None = None,
    marker: str = "o",
) -> None:
    table = Table.read(table_or_path) if isinstance(table_or_path, Path) else table_or_path
    points = astrometry_points(table)
    if len(points["time"]) == 0:
        rax_5.text(0.5, 0.5, "No astrometry", transform=rax_5.transAxes, ha="center", va="center")
        rax_5.set_axis_off()
        rax_lens.set_axis_off()
        return
    model_5p, model_lens = level3_astrometric_models(points, row)
    if model_5p is None and model_lens is None:
        rax_5.text(0.5, 0.5, "No astrometric model", transform=rax_5.transAxes, ha="center", va="center")
        rax_5.set_axis_off()
        rax_lens.set_axis_off()
        return

    residuals_5p = model_5p - points["al"] if model_5p is not None else np.full_like(points["al"], np.nan)
    residuals_lens = model_lens - points["al"] if model_lens is not None else np.full_like(points["al"], np.nan)
    res_for_clip = np.asarray(residuals_lens, dtype=float)
    finite_mask = np.isfinite(points["time"]) & np.isfinite(points["al_err"]) & (points["al_err"] > 0) & np.isfinite(res_for_clip)
    if finite_mask.any():
        median_res = np.nanmedian(res_for_clip[finite_mask])
        mad = np.nanmedian(np.abs(res_for_clip[finite_mask] - median_res))
        mad_sigma = 1.4826 * mad if np.isfinite(mad) else np.nan
        rms_res = np.sqrt(np.nanmean((res_for_clip[finite_mask] - median_res) ** 2))
        sigma = mad_sigma if np.isfinite(mad_sigma) and mad_sigma > 0 else rms_res
        outlier_mask = np.abs(res_for_clip - median_res) > 5.0 * sigma if np.isfinite(sigma) and sigma > 0 else np.zeros_like(res_for_clip, dtype=bool)
    else:
        outlier_mask = np.zeros_like(res_for_clip, dtype=bool)
    keep = finite_mask & ~outlier_mask
    if not np.any(keep):
        keep = np.isfinite(points["time"]) & np.isfinite(points["al_err"]) & (points["al_err"] > 0)

    ts = points["time"][keep]
    errs = points["al_err"][keep]
    res5 = residuals_5p[keep]
    res_lens = residuals_lens[keep]
    x_min, x_max = xlim if xlim is not None else finite_span(ts, (float(np.nanmin(points["time"])), float(np.nanmax(points["time"]))))

    for ax, residuals, title in [
        (rax_5, res5, "5p. model AL residuals"),
        (rax_lens, res_lens, "Microlensing model AL residuals"),
    ]:
        finite = np.isfinite(ts) & np.isfinite(residuals) & np.isfinite(errs)
        ax.scatter(ts[finite], residuals[finite], c=ts[finite], edgecolor="k", cmap=CMAP, marker=marker, lw=1, alpha=1.0, zorder=4, s=40)
        ax.errorbar(ts[finite], residuals[finite], fmt="none", yerr=errs[finite], zorder=3, c="k", snap=False)
        ax.axhline(y=0, ls="--", c="k", lw=2)
        add_event_markers(ax, row, x_min, x_max)
        ax.text(0.02, 0.94, title, transform=ax.transAxes, ha="left", va="top", fontsize=9, bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none", "pad": 1.5})
        ax.set_xlim(x_min, x_max)
        ax.set_ylabel("Residual [mas]")
        max_abs = np.nanmax(np.abs(residuals[finite])) if np.any(finite) else np.nan
        if not np.isfinite(max_abs) or max_abs == 0:
            max_abs = 1.0
        ax.set_ylim(-1.4 * max_abs, 1.4 * max_abs)
    rax_5.legend(fontsize=7, handles=event_marker_handles(), loc="best")
    rax_5.tick_params(labelbottom=False)
    rax_lens.set_xlabel("Time [year]")

In [ ]:
shared_xlim = merged_time_xlim(photometry_time_span(phot), astrometry_time_span(astro))
fig = plt.figure(figsize=(13, 6), constrained_layout=True)
grid = fig.add_gridspec(1, 2, width_ratios=[1, 1.2])
pg = grid[0].subgridspec(2, 1, height_ratios=[3, 1], hspace=0)
ag = grid[1].subgridspec(2, 1, hspace=0.03)
g_ax = fig.add_subplot(pg[0]); g_res_ax = fig.add_subplot(pg[1], sharex=g_ax)
a5_ax = fig.add_subplot(ag[0]); al_ax = fig.add_subplot(ag[1], sharex=a5_ax)
plot_photometry(g_ax, g_res_ax, phot, row, shared_xlim)
plot_astrometry(a5_ax, al_ax, astro, row, shared_xlim)
fig.suptitle(label)
lo, hi = sorted(g_ax.get_ylim())
phot_audit['inside_g_display_window'] = phot_keep & phot.g_mag.between(lo, hi).to_numpy()
summary['photometry_kept_outside_display_window'] = int((phot_keep & ~phot_audit.inside_g_display_window).sum())
summary['g_display_limits_mag'] = [lo, hi]
fig.savefig(OUTPUT_DIR / 'photometry_astrometry.png', dpi=180)
fig.savefig(OUTPUT_DIR / 'photometry_astrometry.pdf')
display(fig)
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].scatter(days_to_decimal_year(phot.g_obs_time.to_numpy(float)[phot_keep]),
                phot.g_mag.to_numpy(float)[phot_keep], s=20, label='All selected G rows')
outside = phot_keep & ~phot_audit.inside_g_display_window.to_numpy()
axes[0].scatter(days_to_decimal_year(phot.g_obs_time.to_numpy(float)[outside]),
                phot.g_mag.to_numpy(float)[outside], marker='x', c='red', s=65, label='Outside main G window')
axes[0].invert_yaxis(); axes[0].set_ylabel('G [mag]'); axes[0].legend(fontsize=8)
for selected, name, colour, marker in [(keep, 'Retained', 'black', 'o'),
                                      (~keep, 'Rejected / nonfinite', 'red', 'x')]:
    show = selected & np.isfinite(points['time']) & np.isfinite(res_lens)
    axes[1].scatter(points['time'][show], res_lens[show], c=colour, marker=marker, label=name)
if np.isfinite(center) and np.isfinite(scale):
    for sign in (-1, 1):
        axes[1].axhline(center + sign*5*scale, ls='--', color='red', alpha=.5)
axes[1].set_ylabel('Lens model − AL [mas]'); axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel('Time [year]'); ax.grid(alpha=.2)
fig.suptitle('Full data range: display limits and astrometric rejections')
fig.savefig(OUTPUT_DIR / 'unclipped_selection_diagnostic.png', dpi=150)
display(fig)
plt.close(fig)

## Save the dataset and provenance

ECSV keeps table metadata and exact IDs; CSV masks reference the original input
row/CCD slot, and retain rejected measurements. The selected photometry export
includes all selected rows, even those outside the main panel's display window.
The manifest records input checksums, release and the embedded source-code hashes.
Clear outputs before sharing this notebook. No credentials are needed here.

In [ ]:
EMBEDDED_SOURCES = {
    'generate_microlensing_viewer.py': '7c9a4643ac0ad7f649ab02c775bd58b90e6db374b257758ae7ca367308803382',
    'ECLIPSE/trajectories.py': '5f10cc29feb48d8c2bb5ec4bfa9460401180135051bc00b17ad8097c3d2ece19',
}

In [ ]:
phot_audit.to_csv(OUTPUT_DIR / 'photometry_all_rows_with_masks.csv', index=False)
phot_audit.loc[phot_keep].to_csv(OUTPUT_DIR / 'photometry_selected.csv', index=False)
ccd_audit.to_csv(OUTPUT_DIR / 'astrometry_all_ccds_with_masks.csv', index=False)
transit_audit.to_csv(OUTPUT_DIR / 'astrometry_all_transits_with_masks.csv', index=False)
transit_audit.loc[transit_audit.kept_for_figure].to_csv(OUTPUT_DIR / 'astrometry_selected_transits.csv', index=False)
astro.write(OUTPUT_DIR / 'astrometry_original.ecsv', format='ascii.ecsv', overwrite=True)
selected_parameters.write(OUTPUT_DIR / 'catalogue_model_parameters.ecsv', format='ascii.ecsv', overwrite=True)
inputs = [PARAMETER_FILE, PHOT_FILE, ASTRO_FILE, INPUT_PROVENANCE,
          PARAMETER_FILE.with_suffix(PARAMETER_FILE.suffix + '.provenance.json')]
if supplement_applied:
    inputs.append(ASTROMETRY_SUPPLEMENT)
if CANDIDATE_FILE.is_file():
    inputs.append(CANDIDATE_FILE)
manifest = {
    'source_id': SOURCE_ID, 'label': label,
    'processed_at_utc': datetime.now(timezone.utc).isoformat(),
    'archive_url': epoch_provenance.get('endpoint'), 'tap_schema': TAP_SCHEMA, 'data_release': DATA_RELEASE,
    'epoch_download_provenance': epoch_provenance, 'catalogue_provenance': catalogue_provenance,
    'archive_retrieval_date': 'not inferred; consult original download records',
    'astrometry_supplement_applied': supplement_applied,
    'inputs': [{'path': str(p), 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in inputs],
    'embedded_source_sha256': EMBEDDED_SOURCES,
    'selection': summary,
    'note': 'Reproduces existing figure masks, not the internal CU7 fitting selection; no refit.',
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': mpl.__version__},
}
def json_safe(value):
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    return None if isinstance(value, float) and not np.isfinite(value) else value
(OUTPUT_DIR / 'provenance_and_selection.json').write_text(json.dumps(json_safe(manifest), indent=2, allow_nan=False) + '\n')
print(f'Figure, diagnostics, masks, data and provenance saved in {OUTPUT_DIR}')